# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id
print('Available RecordSets:')
for rs in metadata.record_sets:
    print(f"@id: {rs.id}", f"    Name: {rs.name}")
    print("  Fields and their @ids:")
    for field in rs.fields:
        print(f"    @id: {field.id}", f"   Name: {field.name} - Data Type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select one or more record sets by their @id for extraction
# Below, we extract data from all record sets
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display columns and first few rows for the first record set (if available)
if len(record_sets) > 0 and record_sets[0] in dataframes:
    primary_rs = record_sets[0]
    print(f"Fields/columns in RecordSet {primary_rs}: {dataframes[primary_rs].columns.tolist()}")
    display(dataframes[primary_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick the first available numeric field from the first record set
import numpy as np

if len(record_sets) > 0 and record_sets[0] in dataframes:
    df = dataframes[primary_rs]
    # Identify numeric fields by data type
    rs_obj = [r for r in metadata.record_sets if r.id == primary_rs][0]
    numeric_fields = [f for f in rs_obj.fields if f.data_type in ["Number", "Float", "Integer"]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        print(f"Using numeric field: {numeric_field_id}")

        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        except Exception:
            print(f"Could not convert field {numeric_field_id} to numeric. Check data type.")

        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-6)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (if available)
        group_fields = [f.id for f in rs_obj.fields if f.data_type == "Text" and f.id != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
    else:
        print(f"No numeric fields found in record set {primary_rs} for EDA. Available columns: {df.columns.tolist()}")
else:
    print("No record sets with data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and record_sets[0] in dataframes and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # Scatter plot with group field (if available)
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used the `mlcroissant` library to programmatically load and explore the FAIR^2 dataset on extracellular vesicle protein abundance from human mast cells. 

- We loaded dataset metadata and listed available record sets and fields using their `@id`s, as recommended for reproducible references.
- We extracted records directly to pandas DataFrames, demonstrated filtering on numeric fields, normalization, and group-wise aggregation.
- Visualization summarized field distributions and potential relationships.

Refer to each entity/column by its `@id` for all further programmatic access, per Croissant/FAIR data standards. Additional custom analyses or ML pipeline preparation can proceed from here.